## 1. Imports & Config

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

## 2. Load Silver Table

In [0]:
df_flights = spark.table("silver_flights")
df_airlines = spark.table("silver_airlines")
df_airports = spark.table("silver_airports")

df_bronze = spark.table("skylens_macro.team1.bronze_flights")

## 3. Data Quality Framework

In [0]:
def run_test(test_name, df, condition, table_name, threshold=0.0, severity="CRITICAL"):
    total = df.count()
    failed = df.filter(~condition).count()
    
    failure_pct = failed / total if total > 0 else 0
    status = "PASS" if failure_pct <= threshold else "FAIL"
    
    return {
        "test_name": test_name,
        "table_name": table_name,
        "layer": "silver",
        "status": status,
        "severity": severity,
        "rows_checked": total,
        "rows_failed": failed,
        "threshold_pct": threshold,
        "failure_pct": failure_pct,
        "run_timestamp": datetime.now()
    }

## 4. Define Tests

In [0]:
tests = []

### Flights Table Tests

#### A. Completeness Checks

In [0]:
tests.append(run_test("flight_date_not_null", df_flights, F.col("flight_date").isNotNull(), "silver_flights"))
tests.append(run_test("airline_not_null", df_flights, F.col("AIRLINE").isNotNull(), "silver_flights"))
tests.append(run_test("origin_not_null", df_flights, F.col("ORIGIN_AIRPORT").isNotNull(), "silver_flights"))
tests.append(run_test("destination_not_null", df_flights, F.col("DESTINATION_AIRPORT").isNotNull(), "silver_flights"))

#### B. Timestamp Validation

In [0]:
tests.append(run_test(
    "scheduled_departure_ts_not_null",
    df_flights,
    F.col("scheduled_departure_ts").isNotNull(),
    "silver_flights"
))

tests.append(run_test(
    "actual_departure_present_if_not_cancelled",
    df_flights,
    (F.col("CANCELLED") == 1) | F.col("actual_departure_ts").isNotNull(),
    "silver_flights"
))

#### C. Delay Logic

In [0]:
tests.append(run_test(
    "delay_category_logic",
    df_flights,
    (
        ((F.col("ARRIVAL_DELAY") <= 0) & (F.col("delay_category") == "On_Time")) |
        ((F.col("ARRIVAL_DELAY").between(1,15)) & (F.col("delay_category") == "Slight")) |
        ((F.col("ARRIVAL_DELAY").between(16,60)) & (F.col("delay_category") == "Moderate")) |
        ((F.col("ARRIVAL_DELAY") > 60) & (F.col("delay_category") == "severe")) |
        (F.col("CANCELLED") == 1)
    ),
    "silver_flights"
))

#### D. Delay Columns NOT NULL

In [0]:
tests.append(run_test(
    "delay_columns_not_null",
    df_flights,
    (
        F.col("WEATHER_DELAY").isNotNull() &
        F.col("AIRLINE_DELAY").isNotNull() &
        F.col("AIR_SYSTEM_DELAY").isNotNull() &
        F.col("SECURITY_DELAY").isNotNull() &
        F.col("LATE_AIRCRAFT_DELAY").isNotNull()
    ),
    "silver_flights"
))

#### E. Derived Columns

In [0]:
tests.append(run_test(
    "total_delay_correct",
    df_flights,
    F.col("total_delay_minutes") ==
    (
        F.col("WEATHER_DELAY") +
        F.col("AIRLINE_DELAY") +
        F.col("AIR_SYSTEM_DELAY") +
        F.col("SECURITY_DELAY") +
        F.col("LATE_AIRCRAFT_DELAY")
    ),
    "silver_flights"
))

tests.append(run_test(
    "taxi_total_correct",
    df_flights,
    F.col("taxi_total_minutes") ==
    (F.col("TAXI_OUT") + F.col("TAXI_IN")),
    "silver_flights"
))

#### F. Time Anomaly Check

In [0]:
tests.append(run_test(
    "time_anomaly_flag_correct",
    df_flights,
    (
        ((F.col("ELAPSED_TIME") <= 0) | (F.col("AIR_TIME") <= 0)) &
        (F.col("is_time_anomaly") == True)
    ) |
    (
        ((F.col("ELAPSED_TIME") > 0) & (F.col("AIR_TIME") > 0)) &
        (F.col("is_time_anomaly") == False)
    ),
    "silver_flights"
))

#### G. Duplicate Check

In [0]:
dup_df = df_flights.groupBy(
    "FLIGHT_NUMBER","ORIGIN_AIRPORT","DESTINATION_AIRPORT","flight_date"
).count().filter("count > 1")

dup_count = dup_df.count()

tests.append({
    "test_name": "no_duplicates_after_dedup",
    "table_name": "silver_flights",
    "layer": "silver",
    "status": "FAIL" if dup_count > 0 else "PASS",
    "severity": "CRITICAL",
    "rows_checked": df_flights.count(),
    "rows_failed": dup_count,
    "threshold_pct": 0.0,
    "failure_pct": dup_count / df_flights.count() if df_flights.count() > 0 else 0,
    "run_timestamp": datetime.now()
})

#### H. Row Count Retention

In [0]:
tests.append({
    "test_name": "silver_vs_bronze_row_count",
    "table_name": "silver_flights",
    "layer": "silver",
    "status": "PASS" if df_flights.count() >= 0.95 * df_bronze.count() else "FAIL",
    "severity": "CRITICAL",
    "rows_checked": df_flights.count(),
    "rows_failed": df_bronze.count() - df_flights.count(),
    "threshold_pct": 0.05,
    "failure_pct": (df_bronze.count() - df_flights.count()) / df_bronze.count(),
    "run_timestamp": datetime.now()
})

### AIRLINES TABLE TESTS

In [0]:
tests.append(run_test(
    "airline_code_not_null",
    df_airlines,
    F.col("airline_code").isNotNull(),
    "silver_airlines"
))

tests.append(run_test(
    "airline_name_not_null",
    df_airlines,
    F.col("AIRLINE").isNotNull(),
    "silver_airlines"
))

#### A. Airline Uniqueness

In [0]:
dup_airlines = df_airlines.groupBy("airline_code").count().filter("count > 1")

tests.append({
    "test_name": "airline_unique",
    "table_name": "silver_airlines",
    "layer": "silver",
    "status": "FAIL" if dup_airlines.count() > 0 else "PASS",
    "severity": "CRITICAL",
    "rows_checked": df_airlines.count(),
    "rows_failed": dup_airlines.count(),
    "threshold_pct": 0.0,
    "failure_pct": 0.0,
    "run_timestamp": datetime.now()
})

### AIRPORTS TABLE TESTS

In [0]:
tests.append(run_test(
    "airport_code_not_null",
    df_airports,
    F.col("airport_code").isNotNull(),
    "silver_airports"
))

tests.append(run_test(
    "airport_name_not_null",
    df_airports,
    F.col("airport_name").isNotNull(),
    "silver_airports"
))

#### A. Lat/Long Range

In [0]:
tests.append(run_test(
    "valid_lat_long",
    df_airports,
    (F.col("LATITUDE").between(-90,90)) &
    (F.col("LONGITUDE").between(-180,180)),
    "silver_airports"
))

#### B. Airport Uniqueness

In [0]:
dup_airports = df_airports.groupBy("airport_code").count().filter("count > 1")

tests.append({
    "test_name": "airport_unique",
    "table_name": "silver_airports",
    "layer": "silver",
    "status": "FAIL" if dup_airports.count() > 0 else "PASS",
    "severity": "CRITICAL",
    "rows_checked": df_airports.count(),
    "rows_failed": dup_airports.count(),
    "threshold_pct": 0.0,
    "failure_pct": 0.0,
    "run_timestamp": datetime.now()
})

## 5. Convert to DataFrame

In [0]:
result_df = spark.createDataFrame(tests)

## 6. Save to Delta Table

In [0]:
result_df.write.format("delta") \
    .mode("append") \
    .saveAsTable("silver_data_quality_log")

## 7. Fail Pipeline (Airflow Integration)

In [0]:
fail_count = result_df.filter(
    (F.col("status") == "FAIL") & (F.col("severity") == "CRITICAL")
).count()

if fail_count > 0:
    raise Exception(f"{fail_count} CRITICAL data quality tests failed!")